# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze**, nettoyer les anomalies, typer fortement les colonnes (dates/timestamps, numériques) et créer vos colonnes métier dérivées.
3. Mettre en œuvre une **jointure distribuée optimisée** avec **Broadcast Hash Join** (`F.broadcast`) sur une table de référence (sans créer de produit cartésien N-M !).
4. Sauvegarder votre table analytique propre dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze & Diagnostic Qualité

### Exercice 1.1 — Profiler les valeurs manquantes, doublons et plages de valeurs
1. Chargez votre (ou vos) table(s) Delta depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` par colonne clé ainsi que le nombre de doublons éventuels sur l'identifiant métier.

In [2]:
# Table Bronze principale : 1 ligne = 1 course Uber ou Lyft (Boston, nov.–déc. 2018)
BRONZE_MAIN_PATH = "/lakehouse/bronze/rides"
df_bronze = spark.read.format("delta").load(BRONZE_MAIN_PATH)

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

# Valeurs nulles ET valeurs texte "NA" (le CSV code les prix inconnus en "NA", que Spark ne voit pas comme NULL)
COLONNES_CLES = ["id", "timestamp", "datetime", "source", "destination", "cab_type", "product_id", "name",
                 "price", "distance", "surge_multiplier", "short_summary", "temperature"]
compteurs = df_bronze.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__null") for c in COLONNES_CLES]
    + [F.sum((F.trim(F.col(c).cast("string")) == "NA").cast("int")).alias(f"{c}__na") for c in COLONNES_CLES]
).first().asDict()
null_report = spark.createDataFrame(
    [(c, dict(df_bronze.dtypes)[c], int(compteurs[f"{c}__null"]), int(compteurs[f"{c}__na"] or 0)) for c in COLONNES_CLES],
    ["colonne", "type_bronze", "nb_null", "nb_texte_NA"],
)
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)

# Doublons sur l'identifiant métier et plages de valeurs
nb_lignes = df_bronze.count()
nb_ids = df_bronze.select("id").distinct().count()
print(f"Lignes : {nb_lignes:,} | identifiants distincts : {nb_ids:,} | doublons sur `id` : {nb_lignes - nb_ids:,}")
df_bronze.select(
    F.min("datetime").alias("premiere_course"), F.max("datetime").alias("derniere_course"),
    F.min(F.col("price").cast("double")).alias("prix_min"), F.max(F.col("price").cast("double")).alias("prix_max"),
    F.min("distance").alias("distance_min"), F.max("distance").alias("distance_max"),
    F.min("surge_multiplier").alias("surge_min"), F.max("surge_multiplier").alias("surge_max"),
).show(vertical=True)

# Qui sont les prix inconnus ?
df_bronze.filter(F.col("price") == "NA").groupBy("cab_type", "name").count().show()

+----------------+-----------+-------+-----------+
|colonne         |type_bronze|nb_null|nb_texte_NA|
+----------------+-----------+-------+-----------+
|id              |string     |0      |0          |
|timestamp       |double     |0      |0          |
|datetime        |timestamp  |0      |0          |
|source          |string     |0      |0          |
|destination     |string     |0      |0          |
|cab_type        |string     |0      |0          |
|product_id      |string     |0      |0          |
|name            |string     |0      |0          |
|price           |string     |0      |55095      |
|distance        |double     |0      |0          |
|surge_multiplier|double     |0      |0          |
|short_summary   |string     |0      |0          |
|temperature     |double     |0      |0          |
+----------------+-----------+-------+-----------+



Lignes : 693,071 | identifiants distincts : 693,071 | doublons sur `id` : 0


-RECORD 0------------------------------
 premiere_course | 2018-11-26 03:40:46 
 derniere_course | 2018-12-18 19:15:10 
 prix_min        | 2.5                 
 prix_max        | 97.5                
 distance_min    | 0.02                
 distance_max    | 7.86                
 surge_min       | 1.0                 
 surge_max       | 3.0                 



+--------+----+-----+
|cab_type|name|count|
+--------+----+-----+
|    Uber|Taxi|55095|
+--------+----+-----+



### Bilan du diagnostic qualité

- **Aucune valeur `NULL`** sur les colonnes clés, mais c'est trompeur : la colonne `price` est typée **texte** (`string`) en Bronze, car le CSV code les prix inconnus par le texte **`"NA"`**. Spark ne le voit pas comme une valeur manquante.
- **55 095 courses sans prix (7,9 %)**. Ce sont **toutes** les courses de l'offre **Uber « Taxi »** : l'API ne donnait pas de prix pour les taxis.
- **Aucun doublon** sur l'identifiant `id` (693 071 lignes = 693 071 identifiants).
- **Plages de valeurs cohérentes :** 3 semaines de données (du 26/11 au 18/12/2018), prix de 2,50 $ à 97,50 $, distances de 0,02 à 7,86 miles (trajets urbains entre 12 quartiers de Boston), majoration (`surge_multiplier`) de ×1 à ×3.
- **Piège découvert sur les dates :** `datetime` et `hour` sont en **heure UTC**, pas en heure de Boston. Le `timestamp` (epoch Unix) 1544777712 correspond à 08:55 UTC, soit **03:55 à Boston** (UTC−5 en hiver). Sans correction, toutes les analyses par heure seraient décalées de 5 heures.

## 2. Nettoyage, Typage Fort & Jointure Optimisée (`BroadcastHashJoin`)

### Rappels d'architecture distribuée :
- **Attention aux jointures N-à-M :** Si vous joignez deux tables de détail partageant une clé non unique, pré-agrégez d'abord à la bonne granularité pour ne pas dupliquer les lignes et fausser vos sommes !
- **Broadcast Hash Join (`F.broadcast(petite_table)`) :** Lorsqu'une table de dimension / nomenclature fait quelques kilo-octets ou méga-octets, la diffuser sur chaque exécuteur supprime totalement l'étape coûteuse de **Shuffle** sur la grande table de faits.
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, enrichir par Broadcast Join et écrire `/lakehouse/silver/<table_silver>`
1. Supprimez les doublons (`dropDuplicates`), filtrez les valeurs techniquement impossibles (ex. durées ou montants négatifs) et convertissez explicitement vos dates (`F.to_timestamp` / `F.to_date`).
2. Créez au moins **2 nouvelles colonnes métier dérivées** utiles pour vos analyses des Jours 3 & 4 (ex. durée écoulée, écart, ratio, tranche horaire, mois `yyyy-MM`, indicateur binaire `0/1`...).
3. Réalisez une jointure avec `F.broadcast(...)` et affichez `.explain("simple")` pour prouver la présence de `BroadcastHashJoin`.
4. Écrivez la table résultante au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [3]:
SILVER_MAIN_PATH = "/lakehouse/silver/rides"           # table de faits : 1 ligne = 1 course propre
SILVER_DIM_PATH = "/lakehouse/silver/dim_vehicules"    # table de référence : 1 ligne = 1 offre de véhicule

# Horodatages manipulés en UTC, puis convertis explicitement en heure de Boston
spark.conf.set("spark.sql.session.timeZone", "UTC")

# --- a) Table de référence (nomenclature) pour pratiquer le Broadcast Join ---
# Le dataset est un seul fichier plat : on en extrait le catalogue des 13 offres
# et on l'enrichit d'une gamme, d'un niveau de gamme (1 = le moins cher) et d'un nombre de places.
GAMMES = {
    "Shared": ("Partagé", 1, 2), "UberPool": ("Partagé", 1, 2),
    "Lyft": ("Standard", 2, 4), "UberX": ("Standard", 2, 4),
    "WAV": ("Accessible PMR", 2, 4), "Taxi": ("Taxi", 2, 4),
    "Lyft XL": ("Grande capacité", 3, 6), "UberXL": ("Grande capacité", 3, 6),
    "Lux": ("Premium", 4, 4), "Lux Black": ("Premium", 4, 4), "Black": ("Premium", 4, 4),
    "Lux Black XL": ("Premium grande capacité", 5, 6), "Black SUV": ("Premium grande capacité", 5, 6),
}
df_gammes = spark.createDataFrame([(k, *v) for k, v in GAMMES.items()], ["name", "gamme", "niveau_gamme", "places_max"])
(
    df_bronze.select("product_id", "cab_type", "name").distinct()
    .join(df_gammes, on="name", how="left")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_DIM_PATH)
)
dim_vehicules = spark.read.format("delta").load(SILVER_DIM_PATH)
dim_vehicules.orderBy("cab_type", "niveau_gamme").show(truncate=False)

# --- b) Nettoyage + typage fort ---
df_clean = (
    df_bronze
    .dropDuplicates(["id"])                                        # doublons sur l'identifiant (0 aujourd'hui, la règle protège les rechargements)
    .withColumn("price", F.col("price").cast("double"))            # "NA" -> NULL, texte -> nombre
    .withColumn("ts_utc", F.col("timestamp").cast("timestamp"))    # epoch Unix -> horodatage (UTC)
    .withColumn("ts_boston", F.from_utc_timestamp("ts_utc", "America/New_York"))  # heure locale réelle
    .filter(F.col("price").isNotNull())                            # courses sans prix (Taxi) : inexploitables
    .filter((F.col("price") > 0) & (F.col("distance") > 0))        # valeurs impossibles
    .withColumn("meteo", F.trim("short_summary"))                  # espaces parasites : " Rain " -> "Rain"
)

# --- c) Colonnes métier dérivées pour les Jours 3 et 4 ---
JOURS = ["dimanche", "lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi"]
heure = F.hour("ts_boston")
df_enrichi = (
    df_clean
    .withColumn("date_course", F.to_date("ts_boston"))
    .withColumn("mois", F.date_format("ts_boston", "yyyy-MM"))
    .withColumn("heure_boston", heure)
    .withColumn("jour_semaine", F.element_at(F.array(*[F.lit(j) for j in JOURS]), F.dayofweek("ts_boston")))
    .withColumn("is_weekend", F.dayofweek("ts_boston").isin(1, 7).cast("int"))
    .withColumn("tranche_horaire",
                F.when(heure < 6, "1. Nuit (0h-6h)")
                 .when(heure < 10, "2. Pointe du matin (6h-10h)")
                 .when(heure < 16, "3. Journée (10h-16h)")
                 .when(heure < 20, "4. Pointe du soir (16h-20h)")
                 .otherwise("5. Soirée (20h-24h)"))
    .withColumn("prix_par_mile", F.round(F.col("price") / F.col("distance"), 2))
    .withColumn("is_surge", (F.col("surge_multiplier") > 1).cast("int"))   # tarif majoré 0/1
    .withColumn("is_pluie", (F.col("precip_intensity") > 0).cast("int"))   # il pleut 0/1
)

# --- d) Jointure optimisée : la petite table (13 lignes) est diffusée à chaque exécuteur ---
COLONNES_SILVER = [
    "id", "ts_utc", "ts_boston", "date_course", "mois", "jour_semaine", "is_weekend", "heure_boston", "tranche_horaire",
    "source", "destination", "cab_type", "name", "gamme", "niveau_gamme", "places_max",
    "price", "distance", "prix_par_mile", "surge_multiplier", "is_surge",
    "meteo", "is_pluie", "temperature", "precip_intensity", "humidity", "wind_speed", "latitude", "longitude",
]
df_silver = (
    df_enrichi
    .join(F.broadcast(dim_vehicules.select("product_id", "gamme", "niveau_gamme", "places_max")),
          on="product_id", how="left")
    .select(*COLONNES_SILVER)
)

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé `df_silver`."
df_silver.explain("simple")

# --- e) Écriture de la table Silver au format Delta ---
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {spark.read.format('delta').load(SILVER_MAIN_PATH).count():,} lignes")
print(f"Lignes retirées par le nettoyage : {df_bronze.count() - spark.read.format('delta').load(SILVER_MAIN_PATH).count():,}")
print("Offres sans gamme après jointure :", spark.read.format('delta').load(SILVER_MAIN_PATH).filter(F.col('gamme').isNull()).count())
spark.read.format("delta").load(SILVER_MAIN_PATH).select(
    "ts_utc", "ts_boston", "tranche_horaire", "source", "destination", "name", "gamme", "price", "prix_par_mile", "meteo"
).show(5, truncate=False)

+------------+------------------------------------+--------+-----------------------+------------+----------+
|name        |product_id                          |cab_type|gamme                  |niveau_gamme|places_max|
+------------+------------------------------------+--------+-----------------------+------------+----------+
|Shared      |lyft_line                           |Lyft    |Partagé                |1           |2         |
|Lyft        |lyft                                |Lyft    |Standard               |2           |4         |
|Lyft XL     |lyft_plus                           |Lyft    |Grande capacité        |3           |6         |
|Lux         |lyft_premier                        |Lyft    |Premium                |4           |4         |
|Lux Black   |lyft_lux                            |Lyft    |Premium                |4           |4         |
|Lux Black XL|lyft_luxsuv                         |Lyft    |Premium grande capacité|5           |6         |
|UberPool    |997ac

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#48, ts_utc#3244, ts_boston#3305, date_course#3430, mois#3494, jour_semaine#3625, is_weekend#3692, heure_boston#3559, tranche_horaire#3760, source#4162, destination#4164, cab_type#4166, name#4170, gamme#2314, niveau_gamme#2315L, places_max#2316L, price#3184, distance#4174, prix_par_mile#3829, surge_multiplier#4176, is_surge#3899, meteo#3367, is_pluie#3970, temperature#4182, ... 5 more fields]
   +- BroadcastHashJoin [product_id#4168], [product_id#2312], LeftOuter, BuildRight, false
      :- Project [id#48, source#4162, destination#4164, cab_type#4166, product_id#4168, name#4170, price#3184, distance#4174, surge_multiplier#4176, latitude#4178, longitude#4180, temperature#4182, precip_intensity#4190, humidity#4194, wind_speed#4196, ts_utc#3244, ts_boston#3305, trim(short_summary#4186, None) AS meteo#3367, cast(ts_boston#3305 as date) AS date_course#3430, date_format(ts_boston#3305, yyyy-MM, Some(UTC)) AS mois#3494, hou

Table Silver écrite dans /lakehouse/silver/rides : 637,976 lignes


Lignes retirées par le nettoyage : 55,095


Offres sans gamme après jointure : 0


+-------------------+-------------------+---------------------------+-----------------+----------------+---------+--------+-----+-------------+-------------+
|ts_utc             |ts_boston          |tranche_horaire            |source           |destination     |name     |gamme   |price|prix_par_mile|meteo        |
+-------------------+-------------------+---------------------------+-----------------+----------------+---------+--------+-----+-------------+-------------+
|2018-11-27 02:15:22|2018-11-26 21:15:22|5. Soirée (20h-24h)        |North End        |Beacon Hill     |Lux Black|Premium |22.5 |12.3         |Rain         |
|2018-12-18 01:15:10|2018-12-17 20:15:10|5. Soirée (20h-24h)        |North Station    |Haymarket Square|UberPool |Partagé |6.5  |11.61        |Mostly Cloudy|
|2018-12-03 08:58:01|2018-12-03 03:58:01|1. Nuit (0h-6h)            |Fenway           |North Station   |Lyft     |Standard|10.5 |3.26         |Foggy        |
|2018-12-14 11:40:11|2018-12-14 06:40:11|2. Pointe d

### Choix de nettoyage et d'enrichissement

| Étape | Règle | Effet |
|---|---|---|
| Doublons | `dropDuplicates(["id"])` | 0 ligne retirée (règle conservée pour les rechargements futurs) |
| Typage | `price` texte → `double` (`"NA"` → `NULL`) ; epoch → `ts_utc` → **`ts_boston`** (`from_utc_timestamp`) | prix calculables, heures justes |
| Valeurs inexploitables | prix `NULL` (Taxi), prix ou distance ≤ 0 | **−55 095 lignes** → **637 976 courses** en Silver |
| Textes | `trim` sur la météo (`" Rain "` → `"Rain"`) | catégories propres pour les regroupements |

**Colonnes métier créées :** `date_course`, `mois`, `jour_semaine`, `is_weekend`, `heure_boston`, `tranche_horaire` (nuit, pointe du matin, journée, pointe du soir, soirée), `prix_par_mile`, `is_surge` (tarif majoré 0/1), `is_pluie` (0/1). Par la jointure : `gamme`, `niveau_gamme` (1 = partagé … 5 = premium grande capacité) et `places_max`.

**Preuve du Broadcast Hash Join :** le plan physique affiche `BroadcastHashJoin [product_id], LeftOuter, BuildRight`. La petite table `dim_vehicules` (13 lignes) est copiée sur chaque exécuteur, donc la grande table (637 976 lignes) n'a pas besoin d'être redistribuée (pas de *shuffle*) pour la jointure. **Pas de produit N-M :** `product_id` est unique dans la table de référence (13 lignes, 13 offres), chaque course trouve exactement une offre. Le nombre de lignes est inchangé et aucune course ne reste sans gamme.

**Narrow vs Wide, Lazy Evaluation :**
- `filter`, `withColumn`, `cast` sont des transformations **narrow** : chaque partition est traitée seule, sans échange de données.
- `dropDuplicates(["id"])` est une transformation **wide** : le plan montre `Exchange hashpartitioning(id, 200)`. C'est un **shuffle** : toutes les lignes d'un même `id` doivent se retrouver dans la même partition pour être comparées.
- Tant qu'aucune **action** n'est appelée (`write`, `count`, `show`), Spark ne lit rien : il construit seulement le plan, c'est la *Lazy Evaluation*. Le plan ci-dessus est affiché par `explain()` avant toute lecture des données. Sur Spark UI (http://localhost:4040), les jobs n'apparaissent qu'au moment des actions.

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [4]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. Test du Schema Enforcement : un DataFrame au mauvais schéma doit être refusé par Delta Lake
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794826031105,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"51059d4c-ce49-4286-a59a-a1a75cc5ab97","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-8d7e03e7-f1b9-44d9-9536-cc9355f183d7-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-94bb43a2-aa81-4888-8586-f037f550016f-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-ea351242-1a5f-4fb8-a0c0-bcbd47bfcfca-c000.snappy.parquet","partition

Schema Enforcement validé — écriture invalide bloquée : [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 51059d4c-ce49-4286-a59a-a1a75cc5ab97).


### Ce que montrent le journal `_delta_log` et le Schema Enforcement

- Le fichier `00000000000000000000.json` est le **journal de la version 0**. Une ligne `commitInfo` (quoi, quand : `WRITE` en mode `Overwrite`), une ligne `metaData` (identifiant et **schéma** de la table), une ligne `protocol`, puis une ligne `add` par fichier Parquet ajouté (**23 fichiers**). Lire une table Delta, c'est d'abord lire ce journal pour savoir quels fichiers forment la version courante.
- **Schema Enforcement :** l'ajout d'un DataFrame aux colonnes inconnues (`fake_id`, `intruder_col`) est **refusé** (`A schema mismatch detected`). Rien n'est écrit : la table reste cohérente. C'est le « C » (cohérence) d'ACID, qu'un simple dossier de fichiers CSV ou Parquet ne garantit pas.

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [5]:
# Scénario : la plateforme envoie un lot correctif -> 5 courses dont le prix est réajusté (+1 $)
# et 1 course arrivée en retard (nouvel identifiant). MERGE = mise à jour si l'id existe, insertion sinon.
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
version_avant = target_delta.history(1).select("version").first()[0]

df_actuel = spark.read.format("delta").load(SILVER_MAIN_PATH)
corrections = (
    df_actuel.orderBy("id").limit(5)
    .withColumn("price", F.round(F.col("price") + 1.0, 2))
    .withColumn("prix_par_mile", F.round(F.col("price") / F.col("distance"), 2))
)
course_en_retard = df_actuel.orderBy("id").limit(1).withColumn("id", F.lit("course-en-retard-0001"))
lot = corrections.unionByName(course_en_retard)

(
    target_delta.alias("t")
    .merge(lot.alias("s"), "t.id = s.id")
    .whenMatchedUpdate(set={"price": "s.price", "prix_par_mile": "s.prix_par_mile"})
    .whenNotMatchedInsertAll()
    .execute()
)

# Time Travel : la version d'avant le MERGE est toujours lisible
ids_lot = [r["id"] for r in lot.select("id").collect()]
avant = spark.read.format("delta").option("versionAsOf", version_avant).load(SILVER_MAIN_PATH)
apres = spark.read.format("delta").load(SILVER_MAIN_PATH)
print(f"Lignes version {version_avant} (avant MERGE) : {avant.count():,} | version courante : {apres.count():,}")
(
    avant.filter(F.col("id").isin(ids_lot)).select("id", F.col("price").alias("prix_avant"))
    .join(apres.filter(F.col("id").isin(ids_lot)).select("id", F.col("price").alias("prix_apres")), "id", "full")
    .orderBy("id").show(truncate=False)
)
print("Version initiale (versionAsOf = 0) :",
      f"{spark.read.format('delta').option('versionAsOf', 0).load(SILVER_MAIN_PATH).count():,} lignes")

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"

Lignes version 0 (avant MERGE) : 637,976 | version courante : 637,977


+------------------------------------+----------+----------+
|id                                  |prix_avant|prix_apres|
+------------------------------------+----------+----------+
|00005b8c-5647-4104-9ac6-94fa6a40f3c3|9.0       |10.0      |
|00006eeb-0183-40c1-8198-c441d3c8a734|3.0       |4.0       |
|00008b42-5ecc-4f66-b4b9-b22a331634e6|27.5      |28.5      |
|000094c0-00c4-43f1-ae1b-4693eec2a580|14.0      |15.0      |
|0000a8b2-e4d3-4227-8374-af8a2366e475|35.0      |36.0      |
|course-en-retard-0001               |NULL      |9.0       |
+------------------------------------+----------+----------+



Version initiale (versionAsOf = 0) : 637,976 lignes


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1|2026-11-16 10:47:25.473|    MERGE|{numTargetRowsCopied -> 140441, numTargetRowsDeleted -> 0, numTargetFilesAdde...|
|      0|2026-11-16 10:47:11.173|    WRITE|           {numFiles -> 23, numOutputRows -> 637976, numOutputBytes -> 39299683}|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+



### Bilan du MERGE INTO et du Time Travel

- **Version 0 (`WRITE`)** : 637 976 courses.
- **Version 1 (`MERGE`)** : le lot correctif a **mis à jour 5 courses** (prix +1 $) et **inséré 1 nouvelle course**, soit **637 977 lignes**. Une seule opération atomique : soit tout le lot passe, soit rien.
- **Time Travel :** avec `versionAsOf = 0`, on relit les **anciens prix** (9,00 $ → 10,00 $, etc.) et la table sans la course en retard. On peut donc auditer ou annuler une modification.
- **Coût caché :** les fichiers Parquet sont immuables. Pour modifier 5 lignes, Delta a **réécrit les 5 fichiers concernés** (140 441 lignes recopiées) et ajouté 22 nouveaux fichiers ; les anciens restent sur disque pour le Time Travel. C'est pourquoi on compacte ensuite avec `OPTIMIZE` (Jour 3).